#### ***Retrieval Evaluation 2***

##### ***Retrieval Evaluation is a process of evaluating the retrieved documents by it's relevant to the user query.***

In [1]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [3]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

C:\Users\mukko\AppData\Local\Temp\ipykernel_2316\2542035176.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader


Number Of Documents: 3983


In [4]:
### Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [5]:
### BM25 Retriever

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)

In [6]:
bm25_retriever.k=10

In [7]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)

In [9]:
similarity_retriever = vectorstore.as_retriever(search_type = "similarity",
        search_kwargs = {"k":10}
)

In [10]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever,bm25_retriever],
    weights=[0.8,0.2]
)

In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [12]:
evaluation_data = [
    {
        "query": "What is a Kubernetes Deployment?",
        "relevant_pages": [5],
    },
    {
        "query": "What is a Kubernetes Pod?",
        "relevant_pages": [83, 85],
    },
    {
        "query": "What is a Kubernetes Service?",
        "relevant_pages": [246, 148, 242],
    },
    {
        "query": "What is a ReplicaSet?",
        "relevant_pages": [156, 164],
    },
    {
        "query": "What is a ConfigMap?",
        "relevant_pages": [392],
    },
    {
        "query": "Deployment spec replicas desired state status",
        "relevant_pages": [5],
    },
    {
        "query": "Pod containers shared network storage node",
        "relevant_pages": [89, 85, 12],
    },
    {
        "query": "Service clusterIP selector endpoints Pods",
        "relevant_pages": [148, 242, 266],
    },
    {
        "query": "How is a Deployment related to a ReplicaSet?",
        "relevant_pages": [156, 163, 130],
    },
    {
        "query": "How does a ReplicaSet maintain Pods?",
        "relevant_pages": [156, 164, 208],
    },
]

In [13]:
def get_doc_page(docs):
    pages = []
    for doc in docs:
        pages.append(doc.metadata.get("page"))
    return pages

In [20]:
### recall: How many retrieved documents are relevant to the number of relevant documents based on user's query.
def recall_at_k(retrieved_pages,relevant_pages):

    relevant_pages = set(relevant_pages)

    retrieved_pages = set(retrieved_pages)
    if not relevant_pages:
        return 0


    return len(retrieved_pages & relevant_pages)/len(relevant_pages)

In [15]:
### Precision; How many retrieved documents are relevant out of top_k based on user's query.
def precision_at_k(retrieved_pages,relevant_pages):

    if not relevant_pages:
        return 0
    counter = 0

    for page in retrieved_pages:
        if page in relevant_pages:
            counter = counter+1

    return counter/len(retrieved_pages)

In [16]:
###MRR(Mean Reciprocal Rank) means how quickly finds the relevant document from the retrieved documents. 
def mrr_at_k(retrieved_pages,relevant_pages):

    if not relevant_pages:
        return 0

    rr = 0
    for rank,page in enumerate(retrieved_pages,start = 1):

        if page in relevant_pages:
            rr = 1/rank
            break

    return rr

In [21]:
for evaluate_data in evaluation_data:
    query = evaluate_data['query']
    relevant_pages = evaluate_data['relevant_pages']
    print("Query:",query)
    print("Relevant Pages:",relevant_pages)
    retrieveer_docs = hybrid_retriever.invoke(query)[:5]
    retrieved_pages = get_doc_page(retrieveer_docs)
    print("Retrieved Pages:",retrieved_pages)
    hybrid_recall = recall_at_k(retrieved_pages,relevant_pages)
    hybrid_precision = precision_at_k(retrieved_pages,relevant_pages)
    hybrid_mrr = mrr_at_k(retrieved_pages,relevant_pages)
    docs = hybrid_retriever.invoke(query)

    pairs = [[query,doc.page_content] for doc in docs]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(docs,scores),key= lambda x:x[1],reverse=True)

    top_docs = [doc for (doc,score) in ranked_docs[:5]]
    top_pages = get_doc_page(top_docs)
    print("Reranked Top Pages:",top_pages)
    rerank_recall = recall_at_k(top_pages,relevant_pages)
    rerank_precision = precision_at_k(top_pages,relevant_pages)
    rerank_mrr = mrr_at_k(top_pages,relevant_pages)
    print("*"*60)
    print("Hybrid Retrieval Evaluation")
    print("Recall:",hybrid_recall)
    print("Precision:",hybrid_precision)
    print("MRR:",hybrid_mrr)
    print("*"*60)
    print("Reranker Retrieval Evaluatiin")
    print("Recall:",rerank_recall)
    print("Precision:",rerank_precision)
    print("MRR:",rerank_mrr)
    print("#"*60)






Query: What is a Kubernetes Deployment?
Relevant Pages: [5]
Retrieved Pages: [7, 6, 12, 5, 9]
Reranked Top Pages: [9, 5, 2, 638, 6]
************************************************************
Hybrid Retrieval Evaluation
Recall: 1.0
Precision: 0.2
MRR: 0.25
************************************************************
Reranker Retrieval Evaluatiin
Recall: 1.0
Precision: 0.2
MRR: 0.5
############################################################
Query: What is a Kubernetes Pod?
Relevant Pages: [83, 85]
Retrieved Pages: [12, 85, 83, 85, 631]
Reranked Top Pages: [83, 12, 631, 85, 85]
************************************************************
Hybrid Retrieval Evaluation
Recall: 1.0
Precision: 0.6
MRR: 0.5
************************************************************
Reranker Retrieval Evaluatiin
Recall: 1.0
Precision: 0.6
MRR: 1.0
############################################################
Query: What is a Kubernetes Service?
Relevant Pages: [246, 148, 242]
Retrieved Pages: [147, 228, 17, 0